# Train the directional SBI model on the crystal-rotation dataset (A100)

Trains `src/scripts/train.py` (GVP-EGNN encoder + native-S² directional/energy posterior) on the grazing-free `rotate_crystal_mode=True` dataset.

**Before you start** — upload the two prepared CSVs to Google Drive (`MyDrive/siimpl_rot/`):
- `siimpl_train.csv` (merged Pool A + Pool B)
- `siimpl_eval_merged.csv`

These are produced by `src/scripts/generate_data.py` + `src/scripts/prepare_data.py` (see the README). The notebook clones the code, stages the CSVs, and trains. The first run preprocesses the CSVs (builds the kNN cache).

**Runtime:** Runtime → Change runtime type → **A100 GPU**.

In [ ]:
# 1. Confirm A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone the code + install deps
import os
GH_TOKEN = os.environ.get('GH_TOKEN', '')   # only needed if the repo is private
REPO = 'github.com/Walsworth-Group/sbi_inverse_problem.git'
url = f'https://{GH_TOKEN + "@" if GH_TOKEN else ""}{REPO}'
%cd /content
!rm -rf sbi_inverse_problem
!git clone --branch siimpl/gvp-egnn-2026-07-03 --single-branch $url sbi_inverse_problem
%cd /content/sbi_inverse_problem
!git log --oneline -1
!pip -q install -r requirements.txt 2>/dev/null; echo done

In [ ]:
# 3. Mount Drive and stage the data into data/siimpl_rot/
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
SRC = '/content/drive/MyDrive/siimpl_rot'
DST = '/content/sbi_inverse_problem/data/siimpl_rot'
os.makedirs(DST, exist_ok=True)
for f in ['siimpl_train.csv', 'siimpl_eval_merged.csv']:
    s = os.path.join(SRC, f)
    assert os.path.exists(s), f'MISSING in Drive: {s}'
    print('copying', f, f'({os.path.getsize(s)/1e9:.2f} GB) ...')
    shutil.copy(s, os.path.join(DST, f))
!ls -la /content/sbi_inverse_problem/data/siimpl_rot/

## 4. Train

Runs the validated architecture (GVP-EGNN hidden 96 / 6 layers). First run preprocesses the CSVs, then trains Stage 1 → Stage 2 → evaluates. If the session drops mid-run, re-run this cell with `--resume` appended.

In [ ]:
%cd /content/sbi_inverse_problem
%env KMP_DUPLICATE_LIB_OK=TRUE
%env PYTHONIOENCODING=utf-8
!python src/scripts/train.py --directional-head --tag ROT_FINAL

In [ ]:
# 5. Save the trained model + eval results back to Drive
import glob, shutil, os
OUT = '/content/drive/MyDrive/siimpl_rot/ROT_FINAL_outputs'
os.makedirs(OUT, exist_ok=True)
res = '/content/sbi_inverse_problem/results/gvp_egnn_v3_siimpl_ROT_FINAL'
for p in glob.glob(res + '/*.pt') + glob.glob(res + '/*.csv') + glob.glob(res + '/*.json'):
    shutil.copy(p, OUT); print('saved', os.path.basename(p))
print('\nOutputs in Drive:', OUT)
!ls -la $OUT